In [1]:
import pandas as pd
import os
import pandas as pd
import numpy as np
import shutil

try:
    from datetime import datetime, timedelta, UTC
except Exception as e:
    from datetime import datetime, timedelta

In [2]:
df = pd.read_csv(f"FEMS Data/Stations/OSCC/SC01/43709.csv")

In [3]:
cols = df.columns.tolist()

In [4]:
cols

['stationName',
 'observationTime',
 'NFDRType',
 'fuelModelType',
 'oneHR_TL_FuelMoisture',
 'tenHR_TL_FuelMoisture',
 'hundredHR_TL_FuelMoisture',
 'thousandHR_TL_FuelMoisture',
 'kbdi',
 'gsi',
 'woodyLFI_fuelMoisture',
 'herbaceousLFI_fuelMoisture',
 'ignitionComponent',
 'energyReleaseComponent',
 'spreadComponent',
 'burningIndex',
 'NFDRQAFlag']

In [5]:
cols[13]

'energyReleaseComponent'

In [6]:
def var_keys(path):
    
    """
    This function returns the variable keys in the RAWS data.
    
    1) path (String) - The full path to the RAWS Data CSV file.
    
    Optional Arguments: None
    
    Returns
    -------
    
    A list of variable keys
    
    Key Indices
    ---------
    
    'StationName' = 0
    'ObservationTime' = 1
    'NFDRType' = 2
    'FuelModelType' = 3
    'OneHR_TL_FuelMoisture' = 4
    'TenHR_TL_FuelMoisture' = 5
    'HundredHR_TL_FuelMoisture' = 6
    'ThousandHR_TL_FuelMoisture' = 7
    'KBDI' = 8
    'GSI' = 9
    'WoodyLFI_FuelMoisture' = 10
    'HerbaceousLFI_fuelMoisture' = 11
    'IgnitionComponent' = 12
    'EnergyReleaseComponent' = 13
    'SpreadComponent' = 14
    'BurningIndex' = 15
    'NFDRQAFlag' = 16
        
    """
    
    df = pd.read_csv(f"{path}")
    
    keys = df.columns.tolist()
    
    return keys

In [7]:
def number_of_psas(gacc_region):
    
    """
    This function returns the number of PSAs for a specific GACC Region
    
    Required Arguments:
    
    1) gacc_region (String) - The 4-letter GACC region identifier. 
    
    GACC Abbreviations
    ------------------
    
    oscc - South Ops
    oncc - North Ops
    nwcc - Northwest Coordination Center
    swcc - Southwest Coordination Center
    nrcc - Northern Rockies Coordination Center
    gbcc - Great Basin Coordination Center
    aicc - Alaska Interagency Coordination Center
    rmcc - Rocky Mountain Coordination Center
    sacc - Southern Area Coordination Center
    eacc - Eastern Area Coordination Center
    
    Optional Arguments: None
    
    Returns
    -------
    
    The number of PSAs    
    """
    
    gacc_region = gacc_region.lower()
    
    num = {
        'oscc':17,
        'oncc':10,
        'aicc':19,
        'nwcc':13,
        'nrcc':14,
        'gbcc':36,
        'swcc':16,
        'rmcc':29,
        'sacc':57,
        'eacc':25
    }
    
    return num[gacc_region]

In [8]:
def get_gacc_prefix(gacc_region):
    
    """
    This function returns the GACC Prefix
    
    Required Arguments:
    
    1) gacc_region (String) - The 4-letter GACC region identifier. 
    
    Optional Arguments: None
    
    Returns
    -------
    
    The filename hosting the RAWS SIG for each GACC.    
    """
    gacc_region = gacc_region.lower()
    
    prefix = {
        'oscc':'SC',
        'oncc':'NC',
        'sacc':'SA',
        'eacc':'EA',
        'gbcc':'GB',
        'nwcc':'NW',
        'rmcc':'RM',
        'swcc':'SW',
        'nrcc':'NR'
        
    }
    
    return prefix[gacc_region]

In [9]:
def get_psa_climatology(gacc_region):
    
    """
    This function generates a climatology for the period specified for a Predictive Services Area at a specific GACC. 
    
    Required Arguments:

    1) gacc_region (String) - The 4-letter abbreviation of the GACC
    
    GACC Abbreviations
    ------------------
    
    oscc - South Ops
    oncc - North Ops
    nwcc - Northwest Coordination Center
    swcc - Southwest Coordination Center
    nrcc - Northern Rockies Coordination Center
    gbcc - Great Basin Coordination Center
    aicc - Alaska Interagency Coordination Center
    rmcc - Rocky Mountain Coordination Center
    sacc - Southern Area Coordination Center
    eacc - Eastern Area Coordination Center

    Optional Arguments: None

    Returns
    ------- 
    
    A CSV file hosting all the data sorted by PSA at: FEMS Data/{gacc_region}/PSA Climo/{folder_name}/{fname}
    """
    
    num_psas = number_of_psas(gacc_region)
    
    gacc_region = gacc_region.upper()

    paths = []
    for psa in range(1, num_psas):
        for i in range(0, len(os.listdir(f"FEMS Data/Station Climo/{gacc_region}/PSA {psa}"))):
            if i == 0:
                path = f"FEMS Data/Station Climo/{gacc_region}/PSA {psa}/AVG"
            if i == 1:
                path = f"FEMS Data/Station Climo/{gacc_region}/PSA {psa}/MAX"
            if i == 2:
                path = f"FEMS Data/Station Climo/{gacc_region}/PSA {psa}/MIN"
            paths.append(path)

    for p in paths:
        for file in os.listdir(f"{p}"):
            try:
                keys = var_keys(f"{p}/{file}")
                break
            except Exception as e:
                pass

        for p in range(0, len(paths)):
            files = []
            
            if os.path.exists(f"{paths[p]}/.ipynb_checkpoints"):
                shutil.rmtree(f"{paths[p]}/.ipynb_checkpoints")
            else:
                pass
                
            for file in os.listdir(paths[p]):
                print(file)
                files.append(file)
    
                f100 = []
                f1000 = []
                erc = []
                bi = []
                sc = []
                ic = []
    
                for i in range(0, len(files)):
                    try:
                        df = pd.read_csv(f"{paths[p]}/{files[i]}")
                        f100.append(df[keys[2]])
                        f1000.append(df[keys[3]])
                        erc.append(df[keys[9]])
                        bi.append(df[keys[11]])
                        sc.append(df[keys[10]])
                        ic.append(df[keys[8]])
                    except Exception as e:
                        pass
    
                f100_vals = pd.DataFrame()
                for i in range(0, len(f100)):
                    f100_vals[f'col{i}'] = f100[i]
    
                f1000_vals = pd.DataFrame()
                for i in range(0, len(f1000)):
                    f1000_vals[f'col{i}'] = f1000[i]
    
                erc_vals = pd.DataFrame()
                for i in range(0, len(erc)):
                    erc_vals[f'col{i}'] = erc[i]
    
                bi_vals = pd.DataFrame()
                for i in range(0, len(bi)):
                    bi_vals[f'col{i}'] = bi[i]    
    
                sc_vals = pd.DataFrame()
                for i in range(0, len(sc)):
                    sc_vals[f'col{i}'] = sc[i]    

                ic_vals = pd.DataFrame()
                for i in range(0, len(ic)):
                    ic_vals[f'col{i}'] = ic[i]   

                f100_vals['min'] = f100_vals.min(axis=1, numeric_only=True, skipna=True)
                f100_vals['avg'] = f100_vals.mean(axis=1, numeric_only=True, skipna=True)
                
                f1000_vals['min'] = f1000_vals.min(axis=1, numeric_only=True, skipna=True)
                f1000_vals['avg'] = f1000_vals.mean(axis=1, numeric_only=True, skipna=True)
    
                erc_vals['max'] = erc_vals.max(axis=1, numeric_only=True, skipna=True)
                erc_vals['avg'] = erc_vals.mean(axis=1, numeric_only=True, skipna=True)
    
                bi_vals['max'] = bi_vals.max(axis=1, numeric_only=True, skipna=True)
                bi_vals['avg'] = bi_vals.mean(axis=1, numeric_only=True, skipna=True)
    
                sc_vals['max'] = sc_vals.max(axis=1, numeric_only=True, skipna=True)
                sc_vals['avg'] = sc_vals.mean(axis=1, numeric_only=True, skipna=True)
    
                ic_vals['max'] = ic_vals.max(axis=1, numeric_only=True, skipna=True)
                ic_vals['avg'] = ic_vals.mean(axis=1, numeric_only=True, skipna=True)
    
            main = pd.DataFrame()
    
            main['f100_min'] = f100_vals['min']
            main['f100_avg'] = f100_vals['avg']
            main['f1000_min'] = f1000_vals['min']
            main['f1000_avg'] = f1000_vals['avg']
            main['erc_max'] = erc_vals['max']
            main['erc_avg'] = erc_vals['avg']
            main['bi_max'] = bi_vals['max']
            main['bi_avg'] = bi_vals['avg']
            main['sc_max'] = sc_vals['max']
            main['sc_avg'] = sc_vals['avg']
            main['ic_max'] = ic_vals['max']
            main['ic_avg'] = ic_vals['avg']
            jdate = np.arange(1, 367, 1)
            main['julian_date'] = jdate

            if p == 0:
                folder_name = 'AVG'
            if p == 1:
                folder_name = 'MAX'
            if p == 2:
                folder_name = 'MIN'

            if os.path.exists(f"FEMS Data/{gacc_region}/PSA Climo/{folder_name}"):
                pass
            else:
                os.makedirs(f"FEMS Data/{gacc_region}/PSA Climo/{folder_name}")
            
            fname = f"zone_{psa}.csv"
            main.to_csv(f"FEMS Data/{gacc_region}/PSA Climo/{folder_name}/{fname}", index=False)
        

In [10]:
get_psa_climatology('oscc')

BENTON_avg.csv
BENTON_max.csv
BENTON_min.csv
OWENS VALLEY_avg.csv
OWENS VALLEY_max.csv
OWENS VALLEY_min.csv
OAK CREEK_avg.csv
OAK CREEK_max.csv
OAK CREEK_min.csv
BENTON_avg.csv
BENTON_max.csv
BENTON_min.csv
OWENS VALLEY_avg.csv
OWENS VALLEY_max.csv
OWENS VALLEY_min.csv
OAK CREEK_avg.csv
OAK CREEK_max.csv
OAK CREEK_min.csv
BENTON_avg.csv
BENTON_max.csv
BENTON_min.csv
OWENS VALLEY_avg.csv
OWENS VALLEY_max.csv
OWENS VALLEY_min.csv
OAK CREEK_avg.csv
OAK CREEK_max.csv
OAK CREEK_min.csv
BENTON_avg.csv
BENTON_max.csv
BENTON_min.csv
OWENS VALLEY_avg.csv
OWENS VALLEY_max.csv
OWENS VALLEY_min.csv
OAK CREEK_avg.csv
OAK CREEK_max.csv
OAK CREEK_min.csv
BENTON_avg.csv
BENTON_max.csv
BENTON_min.csv
OWENS VALLEY_avg.csv
OWENS VALLEY_max.csv
OWENS VALLEY_min.csv
OAK CREEK_avg.csv
OAK CREEK_max.csv
OAK CREEK_min.csv
BENTON_avg.csv
BENTON_max.csv
BENTON_min.csv
OWENS VALLEY_avg.csv
OWENS VALLEY_max.csv
OWENS VALLEY_min.csv
OAK CREEK_avg.csv
OAK CREEK_max.csv
OAK CREEK_min.csv
BENTON_avg.csv
BENTON_max.cs